# Direct Preference Optimization (DPO) with PEFT of Amazon Nova 1.0 using Amazon SageMaker Training Job

## Lab 1 - Data preparation

In this notebook, we are going to prepare the dataset for later on fine-tune Amazon Nova Micro 1.0


***

In [ ]:
%pip install -r requirements.txt

## Prerequisites

If you are going to use Sagemaker in a local environment. You need access to an IAM Role with the required permissions for Sagemaker. You can find [here](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-roles.html) more about it.

In [ ]:
import sagemaker
import boto3

sess = sagemaker.Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

## Prepare the dataset

In this example, we are going to load [nvidia/When2Call](https://huggingface.co/datasets/nvidia/When2Call) dataset, an open-source dataset and model suite focused on enabling and improving function calling capabilities for large language models (LLMs).

In [ ]:
import datasets
from datasets import load_dataset

dataset = (
    load_dataset(
        "nvidia/When2Call",
        "train_pref",
        split="train",
        streaming=True,
    )
    .take(2000)
    .shuffle(buffer_size=500)
)

dataset = datasets.Dataset.from_generator(lambda: dataset, features=dataset.features)

dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

_, val = train_test_split(df, test_size=0.2, random_state=42)
train, test = train_test_split(_, test_size=10, random_state=42)

print("Number of train elements: ", len(train))
print("Number of val elements: ", len(val))
print("Number of test elements: ", len(test))

### Utility functions

The notebook defines utility functions to clean the dataset content by removing prefixes and handling special cases:

```python
def clean_prefix(content):
    # Removes prefixes like "USER:", "ASSISTANT:", etc.
    ...

def clean_message_list(message_list):
    # Cleans message lists from None values and converts to proper format
    ...

def clean_numbered_conversation(message_list):
    # Cleans message lists from None values and converts to proper format
    ...
```

In [ ]:
import json
import re


def clean_prefix(content):
    """Remove prefixes from content, according to Nova data_validator"""
    prefixes = [
        "SYSTEM:",
        "System:",
        "USER:",
        "User:",
        "ASSISTANT:",
        "Assistant:",
        "Bot:",
        "BOT:",
    ]

    # Handle array case (list of content items)
    if hasattr(content, "__iter__") and not isinstance(content, str):
        for i, item in enumerate(content):
            if isinstance(item, dict) and "text" in item:
                text = item["text"]
                if isinstance(text, str):
                    # Clean line by line for multi-line text
                    lines = text.split("\n")
                    cleaned_lines = []
                    for line in lines:
                        cleaned_line = line.strip()
                        for prefix in prefixes:
                            if cleaned_line.startswith(prefix):
                                cleaned_line = cleaned_line[len(prefix) :].strip()
                                break
                        cleaned_lines.append(cleaned_line)
                    item["text"] = "\n".join(cleaned_lines)
        return content

    # Handle string case
    if isinstance(content, str):
        lines = content.split("\n")
        cleaned_lines = []
        for line in lines:
            cleaned_line = line.strip()
            for prefix in prefixes:
                if cleaned_line.startswith(prefix):
                    cleaned_line = cleaned_line[len(prefix) :].strip()
                    break
            cleaned_lines.append(cleaned_line)
        return "\n".join(cleaned_lines)

    return content


def clean_message_list(message_list):
    """Clean message list from None values and convert to list of dicts if needed."""
    if isinstance(message_list, str):
        message_list = json.loads(message_list)

    tmp_cleaned = []
    for msg in message_list:
        new_msg = {}
        for key, value in msg.items():
            if key in ["candidates", "content"]:
                if value is None or str(value).lower() == "None":
                    continue
            new_msg[key] = value
        tmp_cleaned.append(new_msg)

    cleaned = []
    for item in tmp_cleaned:
        if item["role"] == "assistant":
            # Clean prefixes from candidates content
            if "candidates" in item:
                candidates = item["candidates"]
                for candidate in candidates:
                    if isinstance(candidate, dict) and "content" in candidate:
                        content = candidate["content"]
                        for content_item in content:
                            if (
                                isinstance(content_item, dict)
                                and "text" in content_item
                            ):
                                # First clean numbered conversation format
                                text = clean_numbered_conversation(content_item["text"])
                                # Then clean regular prefixes
                                content_item["text"] = clean_prefix(text)
            cleaned.append({"role": item["role"], "candidates": item["candidates"]})
        else:
            content = item["content"]
            for content_item in content:
                if isinstance(content_item, dict) and "text" in content_item:
                    text = clean_numbered_conversation(content_item["text"])
                    content_item["text"] = clean_prefix(text)
            cleaned.append({"role": item["role"], "content": content})

    return cleaned


# Additional function to specifically handle the numbered conversation format
def clean_numbered_conversation(text):
    """Clean numbered conversation format like '1. User: ...'"""
    if not isinstance(text, str):
        return text

    # Pattern to match numbered items with User: or Assistant: prefixes
    pattern = r"(\d+\.\s*)(User:|Assistant:)\s*"

    # Replace the pattern, keeping the number but removing the role prefix
    cleaned_text = re.sub(pattern, r"\1", text)

    return cleaned_text

These functions transform the dataset into the format required by Nova models, handling tool calls and formatting:

```python
def transform_tool_format(tool):
    # Transforms tool format to Nova's expected format
    ...

def extract_toolcall_content(text):
    # Extracts content between <TOOLCALL> tags
    ...

def prepare_dataset(sample):
    # Prepares dataset in the required format for Nova models
    ...

def prepare_dataset_test(sample):
    # Formats test dataset for evaluation
    ...
```

In [ ]:
import json
import re


def transform_tool_format(tool):
    """Transform tool from old format to Nova format."""
    if isinstance(tool, str):
        tool = json.loads(tool)

    return {
        "toolSpec": {
            "name": tool["name"],
            "description": tool["description"],
            "inputSchema": {"json": tool["parameters"]},
        }
    }


def extract_toolcall_content(text):
    """Extract content between <TOOLCALL> tags if present."""
    if isinstance(text, dict):
        if text.get("content"):
            text = text.get("content")

    if "<TOOLCALL>" in text and "</TOOLCALL>" in text:
        pattern = r"<TOOLCALL>(.*?)</TOOLCALL>"
        match = re.search(pattern, text, re.DOTALL)
        if match:
            tool_calls_text = []
            if isinstance(match.group(1), str):
                tools = json.loads(match.group(1))

            for tool_call in tools:
                arguments = (
                    json.loads(tool_call["arguments"])
                    if isinstance(tool_call["arguments"], str)
                    else tool_call["arguments"]
                )
                tool_call_json = {
                    "name": tool_call["name"],
                    "parameters": arguments,
                }
                tool_calls_text.append(json.dumps(tool_call_json))

            return "".join(tool_calls_text)
    return text


def prepare_dataset(sample):
    """Prepare dataset in the required format for Nova models"""
    # Add user messages
    result = {"system": [], "messages": []}

    if isinstance(sample["tools"], str):
        tools = json.loads(sample["tools"]) if sample.get("tools") else []
    else:
        tools = sample["tools"]

    transformed_tools = [transform_tool_format(tool) for tool in tools]

    # Add system message with tools if tools exist
    if transformed_tools:
        system_text = (
            "You may call one or more functions to assist with the user query.\n\n"
            "You are provided with function signatures within <tools></tools> XML tags:\n"
            "<tools>\n"
            f"{json.dumps({"tools": transformed_tools})}\n"
            "</tools>\n\n"
            "For each function call, return a json object with function name and parameters:\n"
            '{"name": function name, "parameters": dictionary of argument name and its value}'
        )
        result["system"] = [{"text": system_text}]

    for message in sample.get("messages", []):
        result["messages"].append(
            {
                "role": message["role"],
                "content": [{"text": extract_toolcall_content(message["content"])}],
            }
        )

    chosen = extract_toolcall_content(sample["chosen_response"])
    rejected = extract_toolcall_content(sample["rejected_response"])

    result["messages"].append(
        {
            "role": "assistant",
            "candidates": [
                {"content": [{"text": chosen}], "preferenceLabel": "preferred"},
                {"content": [{"text": rejected}], "preferenceLabel": "non-preferred"},
            ],
        }
    )

    return result

In [ ]:
def prepare_dataset_test(sample):
    """Parse sample and format it for test dataset."""
    # Process tools upfront
    if isinstance(sample["tools"], str):
        tools = json.loads(sample["tools"]) if sample.get("tools") else []
    else:
        tools = sample["tools"]

    transformed_tools = [transform_tool_format(tool) for tool in tools]

    # Initialize variables
    system_content = ""
    current_input = ""

    # Add system message with tools if tools exist
    if transformed_tools:
        system_content = (
            "You may call one or more functions to assist with the user query.\n\n"
            "You are provided with function signatures within <tools></tools> XML tags:\n"
            "<tools>\n"
            f"{json.dumps({'tools': transformed_tools})}\n"
            "</tools>\n\n"
            "For each function call, return a json object with function name and parameters:\n"
            '{"name": function name, "parameters": dictionary of argument name and its value}'
        )

    for message in sample.get("messages", []):
        if message["role"] == "user":
            current_input = (
                current_input
                + "\n##User: "
                + extract_toolcall_content(message["content"])
            )
        else:
            current_input = (
                current_input
                + "\n##Assistant: "
                + extract_toolcall_content(message["content"])
            )

    if current_input.startswith("\n"):
        current_input = current_input[1:]

    current_input = current_input.strip()

    chosen = extract_toolcall_content(sample["chosen_response"])

    return {"system": system_content, "query": current_input, "response": chosen}

Let's format the dataset by using the prompt style for Amazon Nova:

```
{
    "system": [{"text": Content of the System prompt}],
    "messages": [
        {
            "role": "user",
            "content": ["text": Content of the user prompt]
        },
        {
            "role": "assistant",
            "content": ["text": Content of the answer]
        },
        ...
        {
            "role": "assistant",
            "candidates": [
                {
                    "content": ["text": Content of the answer, "preferenceLabel": "preferred"],
                    "content": ["text": Content of the answer, "preferenceLabel": "non-preferred"]
                }
            ]
        },
    ]
}
```

In [ ]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset = DatasetDict(
    {"train": train_dataset, "val": val_dataset, "test": test_dataset}
)

train_dataset = dataset["train"].map(
    prepare_dataset, remove_columns=train_dataset.features
)

train_dataset = train_dataset.to_pandas()

train_dataset["messages"] = train_dataset["messages"].apply(clean_message_list)

print(train_dataset.iloc[randint(0, len(train_dataset))].to_json())

val_dataset = dataset["val"].map(
    prepare_dataset, remove_columns=val_dataset.features
)

val_dataset = val_dataset.to_pandas()

val_dataset["messages"] = val_dataset["messages"].apply(clean_message_list)

The test dataset will be formatted with the structure below:

```
{
    "system": "Optional - String containing the system prompt, that sets the behavior, role, or personality of the model",
    "query": "String containing the input prompt",
    "response": "String containing the expected model output"
}
```

In [ ]:
test_dataset = dataset["test"].map(
    prepare_dataset_test, remove_columns=test_dataset.features
)

print(test_dataset[randint(0, len(test_dataset) - 1)])

### Upload to Amazon S3

In [ ]:
import boto3

In [ ]:
s3_client = boto3.client("s3")

if default_prefix:
    input_path = f"{default_prefix}/datasets/nova-dpo-peft"
else:
    input_path = f"datasets/nova-dpo-peft"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"
test_dataset_s3_path = f"s3://{bucket_name}/{input_path}/test/gen_qa.jsonl"

In [ ]:
import os

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/val", exist_ok=True)
os.makedirs("./data/test", exist_ok=True)

train_dataset.to_json("./data/train/dataset.jsonl", orient="records", lines=True)
val_dataset.to_json("./data/val/dataset.jsonl", orient="records", lines=True)
test_dataset.to_json("./data/test/gen_qa.jsonl")

s3_client.upload_file(
    "./data/train/dataset.jsonl", bucket_name, f"{input_path}/train/dataset.jsonl"
)

s3_client.upload_file(
    "./data/val/dataset.jsonl", bucket_name, f"{input_path}/val/dataset.jsonl"
)

s3_client.upload_file(
    "./data/test/gen_qa.jsonl", bucket_name, f"{input_path}/test/gen_qa.jsonl"
)

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)
print(test_dataset_s3_path)